# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [ ]:
"""The following generates the Quiz all our models will be evaluated on."""

import itertools
import string
import numpy as np
import yaml

import logging

from ordered_set import OrderedSet
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
load_dotenv("./keys.env", verbose=True)

from smolbench.induction.chromatic import (
    ChromaticIntervalsConfig,
    Prompter,
    get_random_exclusive_quiz,
    anneal_intervals,
)
from smolbench.evals import Marks
from smolbench.evals.openrouter import evaluate

template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. At the end of one's term as $role, they have a ceremony where they hand off the"
    " $role ceremonial sceptre to their successor. The following lists the people who were $role and"
    " the years they were $role:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "Has $color1 handed the sceptre to $color2?"
)

extens_template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. The following lists each year and who was $role"
    " that year:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "During the years $query_years, could $color"
    " have headed the $parade parade every year?"
)


def query_gen(
    labels_to_intervals: Dict[Color, Intervals],
    interval_to_label: Dict[Intervals, Color],
    seed: int,
) -> Dict[str, str]:
    """Generates a series of queries"""
    rng: np.random.Generator = np.random.default_rng(seed)
    # Finds max interval.
    n: int = max(interval[1] for interval in interval_to_label)
    for color, intervals in labels_to_intervals.items():
        # Generates a series of true items.
        for start, end in intervals:
            start, end = np.sort(
                rng.choice(range(start, end + 1), size=2, replace=False)
            )
            yield {"color": color, "start": start, "end": end}, True
        # Generates a false proposition.
        invalid_range: Intervals = anneal_intervals(
            itertools.chain(
                (
                    OrderedSet(interval_to_label.keys())
                    - OrderedSet(itertools.chain(*intervals))
                )
            )
        )
        for start, end in invalid_range:
            start = rng.choice(range(start, end))
            # Binom with p = intervals / n capped at end for a similar-ish
            # distr. to positive accounts.
            end = min(
                end,
                start
                + rng.binomial(
                    end - start + 1,
                    np.mean([len(interval) for interval in interval_to_label]) / n,
                )
                + 1,
            )
            yield {"color": color, "start": start, "end": end}, False


SEED: int = 1776
intens_quiz, extens_quiz = get_random_exclusive_quiz(
    ChromaticIntervalsConfig(
        n=int(12 * 250),
        intervals= 250 // 4,
        colors=45,
        seed=SEED,
    ),
    Prompter(
        template,
        {
            "role": "Twislax",
            "parade": "Gildane",
        },
        query_gen,
        extens_template,
    ),
)

## Prompt Validation

In [2]:
print(intens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists the people who were Twislax and the years they were Twislax:
qo was Twislax on 374 to 454.
Lw was Twislax on 1325 to 1403 and 1729 to 1772.
Gv was Twislax on 454 to 533 and 1617 to 1670.
Wd was Twislax on 838 to 935.
UE was Twislax on 2361 to 2370.
ld was Twislax on 2717 to 2762.
Gt was Twislax on 1137 to 1252.
nR was Twislax on 589 to 636 and 2386 to 2511.
Ef was Twislax on 0 to 181, 2151 to 2205, and 2646 to 2675.
wJ was Twislax on 816 to 838, 1860 to 186

In [3]:
print(extens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists each year and who was Twislax that year:
Year 0: Ef.
Year 1: Ef.
Year 2: Ef.
Year 3: Ef.
Year 4: Ef.
Year 5: Ef.
Year 6: Ef.
Year 7: Ef.
Year 8: Ef.
Year 9: Ef.
Year 10: Ef.
Year 11: Ef.
Year 12: Ef.
Year 13: Ef.
Year 14: Ef.
Year 15: Ef.
Year 16: Ef.
Year 17: Ef.
Year 18: Ef.
Year 19: Ef.
Year 20: Ef.
Year 21: Ef.
Year 22: Ef.
Year 23: Ef.
Year 24: Ef.
Year 25: Ef.
Year 26: Ef.
Year 27: Ef.
Year 28: Ef.
Year 29: Ef.
Year 30: Ef.
Year 31: Ef.
Year 32: Ef.
Y

# Decoder-Only Model
This section tests classical decoder-only models.

In [4]:
decode_intens_eval: Marks = evaluate(
    intens_quiz, "mistralai/devstral-small", SEED
)

In [5]:
decode_extens_eval: Marks = evaluate(
    extens_quiz, "mistralai/devstral-small", SEED
)

In [6]:
# Prints out results.
print(
    decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid
)
print(
    decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid
)
# Serializes results
with open('decode_intens.yaml', 'w') as file:
    yaml.dump(
        decode_intens_eval, file, default_flow_style=False, indent=4
    )
with open('decode_extens.yaml', 'w') as file:
    yaml.dump(
        decode_extens_eval, file, default_flow_style=False, indent=4
    )

70 37 0
97 10 0


## CoT Model
This section tests a CoT model.

In [7]:
cot_intens_eval: Marks = evaluate(
    intens_quiz, "google/gemma-3-27b-it", SEED, extra_args={"reasoning": {"enabled": True}}
)

In [8]:
cot_extens_eval: Marks = evaluate(
    extens_quiz, "google/gemma-3-27b-it", SEED, extra_args={"reasoning": {"enabled": True}}
)

In [9]:
print(cot_intens_eval.correct, cot_intens_eval.incorrect, cot_intens_eval.invalid)
print(cot_extens_eval.correct, cot_extens_eval.incorrect, cot_extens_eval.invalid)
# Serializes results
with open('cot_intens.yaml', 'w') as file:
    yaml.dump(
        cot_intens_eval, file, default_flow_style=False, indent=4
    )
with open('cot_extens.yaml', 'w') as file:
    yaml.dump(
        cot_extens_eval, file, default_flow_style=False, indent=4
    )

66 41 0
95 12 0


## MoE Model
This section tests an MoE model.

In [10]:
moe_intens_eval: Marks = evaluate(intens_quiz, "qwen/qwen3-30b-a3b-instruct-2507", SEED)

In [11]:
moe_extens_eval: Marks = evaluate(extens_quiz, "qwen/qwen3-30b-a3b-instruct-2507", SEED)

In [12]:
print(moe_intens_eval.correct, moe_intens_eval.incorrect, moe_intens_eval.invalid)
print(moe_extens_eval.correct, moe_extens_eval.incorrect, moe_extens_eval.invalid)
# Serializes results
with open('moe_intens.yaml', 'w') as file:
    yaml.dump(
        moe_intens_eval, file, default_flow_style=False, indent=4
    )
with open('moe_extens.yaml', 'w') as file:
    yaml.dump(
        moe_extens_eval, file, default_flow_style=False, indent=4
    )

74 33 0
94 13 0


# HRM Model
This section tests an HRM model.

In [ ]:
# hrm_intens_eval: Marks = evaluate(intens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [14]:
# hrm_extens_eval: Marks = evaluate(extens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [15]:
# print(hrm_intens_eval.correct, hrm_intens_eval.incorrect, hrm_intens_eval.invalid)
# print(hrm_extens_eval.correct, hrm_extens_eval.incorrect, hrm_extens_eval.invalid)
# # Serializes results
# with open('hrm_intens.yaml', 'w') as file:
#     yaml.dump(
#         hrm_intens_eval, file, default_flow_style=False, indent=4
#     )
# with open('cot_extens.yaml', 'w') as file:
#     yaml.dump(
#         hrm_extens_eval, file, default_flow_style=False, indent=4
#     )